# Wildfire Risk Prediction

**Goal:** Train a model to predict a property's wildfire risk from its location, using Census socioeconomic data as input features and proximity to historical fire detections as the target.

**Business context:** *[TODO: Write 2-3 sentences on why this matters — insurance underwriting, real estate valuation, urban planning. Who would use this model and how?]*

**Approach:**
1. Generate geographically diverse property locations across CONUS
2. Enrich each property with ~850 Census ACS5 features (income, housing, demographics)
3. Calculate proximity to NASA FIRMS satellite fire detections (VIIRS)
4. Train XGBoost and RandomForest regressors to predict `nearest_fire_km`

**Limitations:** Current run uses county-level Census granularity due to hardware constraints. Tract or block-group level would capture finer neighborhood variation and likely improve results.

In [ ]:
import pandas as pd
import load_wildfires
import load_census
import load_properties
import gis
import train
import visualize
import sqlalchemy as s

from pathlib import Path
from sql_funcs import SQL

from settings import PATH_DATA

In [ ]:
SQL.kill_idle(True)
sql_obj = SQL()


# Load Properties

## Select Properties of Interest

Chosing 300000 properties randomly from US addresses. We will join relevant census data to these addresses. This will probably take awhile, so best to run it overnight.

We don't care about the address itself. We add a census identifier called the GEOID which based on the coordinate's state, county, and tract number.

Using a package that makes use of the [US Census Geocoder API](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/census-geocoder.html), requests can be in batches of 10,000.

https://pypi.org/project/random-address/

In [ ]:
props = load_properties.Properties(sql_obj=sql_obj)
properties = props.get_properties_gpd()
cur_count = properties.shape[0]
desired_count = 300000
if cur_count < desired_count:
    diff = desired_count - cur_count
    print(f"Adding {diff} more properties...")
    props.add_random_properties_geo_first(diff)


# Load Features from US Census

2023 US Census Data

Using an API key, we will use the 'census' Python package to interact with the US Govermnent's census API.

In [ ]:
import geopandas as gpd

MERGED_PARQUET = PATH_DATA / "merged_properties_census.parquet"

if MERGED_PARQUET.exists():
    print(f"Loading pre-merged data from {MERGED_PARQUET}...")
    combined_gdf = gpd.read_parquet(MERGED_PARQUET)
    print(f"Loaded {len(combined_gdf)} properties with census features.")
else:
    print("No pre-merged file found. Fetching census data from API...")
    census = load_census.CensusData(sql_obj=sql_obj, year=2023, granularity='county')
    combined_gdf = census.merge_census_info(properties)

combined_gdf.head()

# Load Wildfire Data

Wildfire detections from NASA's [FIRMS](https://firms.modaps.eosdis.nasa.gov/) system, using the Visible Infrared Imaging Radiometer Suite (VIIRS) aboard NOAA-21 and Suomi NPP satellites. Each detection is a ~375m pixel with fire radiative power (FRP) in megawatts.

Raw detections are clustered spatiotemporally (DBSCAN: 750m spatial, 3-day temporal) to deduplicate overlapping satellite passes into discrete fire events.

In [ ]:
TARGETS_PARQUET = PATH_DATA / "targets_features.parquet"

if TARGETS_PARQUET.exists():
    print(f"Loading pre-computed targets+features from {TARGETS_PARQUET}...")
    targets_features = gpd.read_parquet(TARGETS_PARQUET)
    # Reconstruct proximity column list for drop_cols later
    census_cols = [c for c in targets_features.columns if c.startswith("B")]
    proximity_cols = [c for c in targets_features.columns
                      if c not in census_cols and c not in ["geometry", "geoid"]]
    print(f"Loaded {len(targets_features)} rows x {targets_features.shape[1]} cols "
          f"({len(proximity_cols)} proximity features)")
else:
    print("No pre-computed file found. Running wildfire ETL inline...")
    wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)
    proximity_features = gis.calc_all_features_parallel(combined_gdf, wildfires.data, n_jobs=5)
    targets_features = pd.concat([combined_gdf, proximity_features], axis=1)
    proximity_cols = list(proximity_features.columns)

targets_features.head()

## Proximity Features

Each property gets 12 distance-based features measuring its relationship to nearby wildfires:

| Feature | Description |
|---------|-------------|
| `nearest_fire_km` | Distance to the closest fire event (prediction target) |
| `kde_density` | Kernel density estimate of local fire activity |
| `idw_score` | Inverse-distance-weighted fire intensity within 50 miles |
| `exp_decay_score` | Exponential decay score (captures both proximity and density) |
| `fire_count_10km/25km/50km/100km` | Number of fires within distance rings |
| `fire_FRP_10km/25km/50km/100km` | Cumulative fire radiative power within rings |

In [ ]:
targets_features.head()

# Exploratory Data Analysis

*[TODO: Write 2-3 sentences summarizing what you observed in the EDA. What patterns stand out? Any surprises in the data distributions or missingness?]*

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Target distribution
axes[0].hist(targets_features["nearest_fire_km"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Distance to Nearest Fire (km)")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Distribution")

# 2. Missing data summary by feature group
census_cols_all = [c for c in targets_features.columns if c.startswith("B")]
missing_pct = targets_features[census_cols_all].isna().mean() * 100
axes[1].hist(missing_pct[missing_pct > 0], bins=20, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("% Missing")
axes[1].set_ylabel("Number of Features")
axes[1].set_title(f"Missingness Across {len(census_cols_all)} Census Features")

# 3. Top 10 correlations with target
numeric_cols = targets_features.select_dtypes(include="number").columns
corrs = targets_features[numeric_cols].corrwith(targets_features["nearest_fire_km"]).drop("nearest_fire_km", errors="ignore")
top_corrs = corrs.abs().nlargest(10)
top_corrs.sort_values().plot.barh(ax=axes[2], color="steelblue")
axes[2].set_xlabel("|Correlation| with Target")
axes[2].set_title("Top 10 Correlated Features")

plt.tight_layout()
plt.savefig("figures/eda_summary.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDataset: {targets_features.shape[0]} properties, {targets_features.shape[1]} columns")
print(f"Census features: {len(census_cols_all)}")
print(f"Features with missing data: {(missing_pct > 0).sum()}")
print(f"Target range: {targets_features['nearest_fire_km'].min():.1f} – {targets_features['nearest_fire_km'].max():.1f} km")

# Machine Learning Considerations

## Scoring Method

Using RMSE (Root Mean Squared Error) as the primary metric. Its quadratic penalty on large errors aligns with the business case: underestimating wildfire risk (predicting a property is far from fires when it isn't) is costly for an insurance company.

## Target Variable

`nearest_fire_km` — the distance in kilometres to the nearest wildfire detection. At the current 119-property test scale, this target has limited geographic diversity. A full 300k-property run would provide the variance needed for meaningful model discrimination.

With a larger dataset, `exp_decay_score` (which captures both proximity and density of nearby fires) would be a stronger target choice.

# Preprocessing and Training

## Split Data into Features/Targets

We use `nearest_fire_km` as the target — the distance in kilometres to the nearest wildfire detection. With the full 300k-property dataset, `exp_decay_score` (which captures both proximity and density of nearby fires) would be a better choice, but the small 119-property test set has nearly zero variance in decay score because all properties are ~360 km from the nearest fire.

All other proximity-derived columns are dropped so the model only sees census features as inputs.

In [ ]:
TARGET_COL = "nearest_fire_km"

# All proximity features are derived from the same wildfire data — drop them
# so the model only sees census features as inputs.
drop_cols = ["geometry", "geoid"] + [c for c in proximity_cols if c != TARGET_COL]

# Preprocessing with adaptive imputation based on missingness analysis
# - MCAR features: SimpleImputer (median) - fast
# - MAR features: IterativeImputer - preserves correlations
X_train, X_test, y_train, y_test, feature_names, pipeline = train.preprocess_with_cache(
    targets_features,
    TARGET_COL,
    drop_cols,
    nan_threshold=0.45,
    corr_threshold=0.85,
    mar_corr_threshold=0.1,
    use_cache=True,
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: {y_train.min():.4f} – {y_train.max():.4f}")
print(f"Target std:   {y_train.std():.6f}")
print(f"Target mean:  {y_train.mean():.4f}")

In [ ]:
# Diagnostic: check if the target has meaningful variance
cv = y_train.std() / y_train.mean() * 100  # coefficient of variation
print(f"Coefficient of variation: {cv:.4f}%")
if cv < 1.0:
    print(
        f"WARNING: Target has near-zero variance (CV={cv:.4f}%). "
        f"All properties are ~{y_train.mean():.1f} km from the nearest fire. "
        f"Models will appear to have perfect accuracy but are not learning meaningful patterns. "
        f"Scale to 300k properties for geographic diversity."
    )

## Preprocessing

Drop high-NaN columns, remove correlated features, then impute + scale. All steps are fit on training data only to prevent leakage.

**Adaptive Imputation:** The preprocessing analyzes each feature's missingness mechanism:
- **MCAR** (Missing Completely At Random): Uses `SimpleImputer(median)` - fast, O(n)
- **MAR** (Missing At Random): Uses `IterativeImputer` - preserves feature correlations

**Caching:** Results are cached to `data/cache/preprocess_{hash}.pkl`. Re-runs with unchanged data load from cache in <1s.

## Model Training


### RandomForestRegressor


In [ ]:
rfr_search = train.train_random_forest(X_train, y_train, n_iter=20, cv=5)
print(f"Best RF params: {rfr_search.best_params_}")

rfr_metrics = train.evaluate_model(rfr_search.best_estimator_, X_train, X_test, y_train, y_test)
print(f"RF Train RMSE: {rfr_metrics['train_rmse']:.8f}")
print(f"RF Test  RMSE: {rfr_metrics['test_rmse']:.8f}")

### XGBoost


In [ ]:
xgb_search = train.train_xgboost(X_train, y_train, n_iter=20, cv=5)
print(f"Best XGB params: {xgb_search.best_params_}")

xgb_metrics = train.evaluate_model( xgb_search.best_estimator_,
                                   X_train, X_test, y_train, y_test)
print(f"XGB Train RMSE: {xgb_metrics['train_rmse']:.8f}")
print(f"XGB Test  RMSE: {xgb_metrics['test_rmse']:.8f}")


## Model Comparison

*[TODO: Write 2-3 sentences comparing the models. Which won and why? Is the gap meaningful? What does the train/test RMSE gap tell you about overfitting?]*

In [ ]:
# Side-by-side model comparison
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def full_metrics(model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    return {
        "Train RMSE": np.sqrt(((y_train - y_pred_train) ** 2).mean()),
        "Test RMSE": np.sqrt(((y_test - y_pred_test) ** 2).mean()),
        "Test MAE": mean_absolute_error(y_test, y_pred_test),
        "Test R²": r2_score(y_test, y_pred_test),
    }

comparison = pd.DataFrame({
    "RandomForest": full_metrics(rfr_search.best_estimator_, X_train, X_test, y_train, y_test),
    "XGBoost": full_metrics(xgb_search.best_estimator_, X_train, X_test, y_train, y_test),
})
comparison

## Feature Importance

*[TODO: Write 2-3 sentences interpreting the top features. What do the most important census features tell you about which neighborhoods are closer to wildfires? Do the results make intuitive sense?]*

In [ ]:
#Pick the better model
if xgb_metrics["test_rmse"] <= rfr_metrics["test_rmse"]:
    best_model = xgb_search.best_estimator_
    print("Best model: XGBoost")
else:
    best_model = rfr_search.best_estimator_
    print("Best model: RandomForest")
feature_names = [load_census.census_code_to_label(x) for x in feature_names]

top_features = train.extract_feature_importance(best_model, feature_names, top_n=10)
print(f"\nTop 10 features:\n{top_features}")

In [ ]:
# Feature importance bar chart
fig_importance = visualize.plot_feature_importance(
    top_features, 
    title="Top 10 Feature Importances",
    save_path=Path("figures/feature_importance.png")
)
fig_importance

In [ ]:
# Actual vs Predicted scatter plot
y_pred = best_model.predict(X_test)

fig_scatter = visualize.plot_actual_vs_predicted(
    y_test.values, 
    y_pred,
    title="Actual vs Predicted (Test Set)",
    xlabel="Actual Distance to Fire (km)",
    ylabel="Predicted Distance to Fire (km)",
    save_path=Path("figures/actual_vs_predicted.png")
)
fig_scatter

In [ ]:
model_path = Path("Models") / "best_model.pkl"
train.save_model(best_model, model_path, pipeline=pipeline, feature_names=feature_names)
print(f"Model saved to {model_path}")

# Conclusion

*[TODO: Write your conclusion. Cover these points:]*

*1. **What did the model learn?** Can census features predict wildfire proximity? How well?*

*2. **Key findings:** Which feature groups mattered most? Any surprising predictors?*

*3. **Limitations:** Small sample size (119 properties), county-level granularity, single year of fire data, proximity != actual risk*

*4. **Business recommendation:** How would you use this model? What confidence level is needed before acting on predictions?*

*5. **Next steps:** Scale to 300k properties, add vegetation/weather/topography features, try finer geographic granularity, add a neural network model for comparison*